# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashfiqmahi/assignment_FLyRank-AI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub pandas
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
df = con.sql("""
WITH monthly AS (
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions,
        SUM(gsc_clicks)      FILTER (WHERE gsc_data_available IS TRUE) AS clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS sum_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.impressions,
    CASE WHEN m.impressions > 0 THEN m.clicks * 100.0 / m.impressions ELSE NULL END AS ctr_pct,
    CASE WHEN m.impressions > 0 THEN m.sum_position * 1.0 / m.impressions ELSE NULL END AS avg_position,
    DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS days_since_created,
    d.content_type
FROM monthly m
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON m.content_hash_id = d.content_hash_id
""").df()
df["days_since_created"] = df["days_since_created"].clip(lower=0)


df["position_tier"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["top_3", "page_1 (4-10)", "page_2 (11-20)", "deep (20+)"]
)
# Benchmark CTR per tier, taken directly from Signal 2 weighted results
benchmark_ctr = {
    "top_3": 0.388,
    "page_1 (4-10)": 0.325,
    "page_2 (11-20)": 0.316,
    "deep (20+)": 0.136,
}
df["benchmark_ctr_pct"] = df["position_tier"].map(benchmark_ctr).astype(float)
# The rule, in code
df["visible_flag"] = (df["impressions"] >= 200).astype(int)
df["weak_ctr_flag"] = (df["ctr_pct"] < df["benchmark_ctr_pct"]).astype(int)
df["score"] = df["visible_flag"] * df["weak_ctr_flag"] * df["impressions"].fillna(0)
df["reason_code"] = np.where(
    (df["visible_flag"] == 1) & (df["weak_ctr_flag"] == 1),
    "weak_ctr_for_position",
    "no_action"
)
df["action"] = np.where(df["score"] > 0, "review", "monitor")
import os
os.makedirs("work/outputs", exist_ok=True)
df_sorted = df.sort_values("score", ascending=False)
df_sorted.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Queue written: {len(df_sorted):,} rows")
print(f"Pages flagged for review: {(df_sorted['action']=='review').sum():,}")
df_sorted[["content_hash_id","position_tier","impressions","ctr_pct",
           "benchmark_ctr_pct","score","reason_code","action"]].head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue written: 331,437 rows
Pages flagged for review: 57,496


,content_hash_id,position_tier,impressions,ctr_pct,benchmark_ctr_pct,score,reason_code,action
316206,content_e8a52cf3d5988c07,page_2 (11-20),244931.0,0.273138,0.316,244931.0,weak_ctr_for_position,review
68406,content_0e03de7680314cd5,top_3,221310.0,0.325336,0.388,221310.0,weak_ctr_for_position,review
182485,content_44f34c0a90047651,top_3,212404.0,0.011299,0.388,212404.0,weak_ctr_for_position,review
68371,content_8d7d99f109e19aa2,top_3,203497.0,0.142017,0.388,203497.0,weak_ctr_for_position,review
151098,content_36e53e9c707674fc,deep (20+),194579.0,0.124371,0.136,194579.0,weak_ctr_for_position,review
261823,content_b99ea6861864dea5,page_1 (4-10),194337.0,0.185760,0.325,194337.0,weak_ctr_for_position,review
68401,content_4ffe18112a5642e3,top_3,186983.0,0.313397,0.388,186983.0,weak_ctr_for_position,review
96029,content_acbcc847f8996314,page_1 (4-10),170808.0,0.153389,0.325,170808.0,weak_ctr_for_position,review
3677,content_471d9cabce329a66,page_1 (4-10),164885.0,0.240167,0.325,164885.0,weak_ctr_for_position,review
278646,content_fd2117c2c6790e4b,page_1 (4-10),151166.0,0.269902,0.325,151166.0,weak_ctr_for_position,review


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am using k-means clustering. I fits because this method does not need any answer key. Since clustering is a unsupervised learning and no one has already ranked the pages so K-means clustering is perfect toolkit. I am clustering impressions, ctr_pct, avg_position, days_since_created, content_type. We have to standardize every numeric feature before clustering because some feature's numeric value is huge and some are very low. So the features with larger value may dominate.

In [2]:
# This is the evidence behind the "we must standardize" claim above.
features_preview = df[["impressions", "ctr_pct", "avg_position", "days_since_created"]]
print(features_preview.describe().T[["mean", "std", "min", "max"]])


                           mean          std  min       max
impressions         1587.986675  5431.337724  1.0  617124.0
ctr_pct                0.459397     3.775992  0.0     100.0
avg_position          15.992270    18.097575  0.0     309.0
days_since_created   207.806307   120.531290  0.0     494.0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Though we are doing clustering which has no labeled dataset, we still spliting the dataset using client_id. We split them into two groups, find clusters on one group, then check if those same-shaped clusters show up in the other group.
I choose grouped by client_id because pages from same client are not independent (same industry, same writing style). If we split randomly page-by-page, a client's pages could end up half in "training" and half in "testing"


In [3]:
from sklearn.model_selection import GroupShuffleSplit

# Grouped split: all of one client's pages stay on the same side
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, holdout_idx = next(gss.split(df, groups=df["client_hash_id"]))

df_train = df.iloc[train_idx].copy()
df_holdout = df.iloc[holdout_idx].copy()

print(f"Train: {len(df_train):,} pages from {df_train['client_hash_id'].nunique()} clients")
print(f"Holdout: {len(df_holdout):,} pages from {df_holdout['client_hash_id'].nunique()} clients")

# Sanity check: zero overlap in clients between the two sides
overlap = set(df_train['client_hash_id']) & set(df_holdout['client_hash_id'])
print(f"Clients appearing in both sides (should be 0): {len(overlap)}")

Train: 281,614 pages from 38 clients
Holdout: 49,823 pages from 17 clients
Clients appearing in both sides (should be 0): 0


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# Check how big the missing-value problem actually is
print("Missing values in df_train:")
print(df_train[["ctr_pct", "avg_position"]].isna().sum())
print(f"\nOut of {len(df_train):,} total training rows")

# Filter: keep only pages with real search visibility (has impressions > 0)
df_train_c = df_train[df_train["impressions"] > 0].copy()
df_holdout_c = df_holdout[df_holdout["impressions"] > 0].copy()

print(f"\nAfter filtering to visible pages:")
print(f"Train: {len(df_train_c):,} pages ({len(df_train_c)/len(df_train):.1%} kept)")
print(f"Holdout: {len(df_holdout_c):,} pages ({len(df_holdout_c)/len(df_holdout):.1%} kept)")


Missing values in df_train:
ctr_pct         132566
avg_position    132566
dtype: int64

Out of 281,614 total training rows

After filtering to visible pages:
Train: 149,048 pages (52.9% kept)
Holdout: 27,690 pages (55.6% kept)


In [7]:
# Recreate your Week 4 baseline rule, applied to the same filtered rows
def add_baseline(d):
    d = d.copy()
    d["position_tier"] = pd.cut(
        d["avg_position"], bins=[0, 3, 10, 20, 1000],
        labels=["top_3", "page_1 (4-10)", "page_2 (11-20)", "deep (20+)"]
    )
    benchmark_ctr = {"top_3": 0.388, "page_1 (4-10)": 0.325, "page_2 (11-20)": 0.316, "deep (20+)": 0.136}
    d["benchmark_ctr_pct"] = d["position_tier"].map(benchmark_ctr).astype(float)
    d["visible_flag"] = (d["impressions"] >= 200).astype(int)
    d["weak_ctr_flag"] = (d["ctr_pct"] < d["benchmark_ctr_pct"]).astype(int)
    d["action"] = np.where((d["visible_flag"] == 1) & (d["weak_ctr_flag"] == 1), "review", "monitor")
    return d

df_train_c = add_baseline(df_train_c)

# THE comparison table: how does the old rulebook's verdict split across your NEW clusters?
comparison = pd.crosstab(df_train_c["cluster"], df_train_c["action"], normalize="index").round(3)
comparison["n_pages"] = df_train_c.groupby("cluster").size()
print(comparison)

action   monitor  review  n_pages
cluster                          
0          0.671   0.329    65592
1          0.668   0.332    61846
2          0.329   0.671     2464
3          1.000   0.000      335
4          0.812   0.188    18811


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

cluster_features = ["impressions", "ctr_pct", "avg_position", "days_since_created"]

# 1. Scale — fit on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(df_train_c[cluster_features])
X_holdout_scaled = scaler.transform(df_holdout_c[cluster_features])

# 2. Fit final model ONCE
FINAL_K = 5
kmeans_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
df_train_c["cluster"] = kmeans_final.fit_predict(X_train_scaled)
df_holdout_c["cluster"] = kmeans_final.predict(X_holdout_scaled)

# 3. Profile both sides, side by side, in the SAME cell
train_profile = df_train_c.groupby("cluster")[cluster_features].median()
train_profile["n_pages"] = df_train_c.groupby("cluster").size()

holdout_profile = df_holdout_c.groupby("cluster")[cluster_features].median()
holdout_profile["n_pages"] = df_holdout_c.groupby("cluster").size()

print("=== TRAIN profile ===")
print(train_profile)
print("\n=== HOLDOUT profile ===")
print(holdout_profile)

# 4. Baseline comparison, computed right here, same cell
df_train_c = add_baseline(df_train_c)
df_holdout_c = add_baseline(df_holdout_c)

train_comparison = pd.crosstab(df_train_c["cluster"], df_train_c["action"], normalize="index").round(3)
train_comparison["n_pages"] = df_train_c.groupby("cluster").size()

holdout_comparison = pd.crosstab(df_holdout_c["cluster"], df_holdout_c["action"], normalize="index").round(3)
holdout_comparison["n_pages"] = df_holdout_c.groupby("cluster").size()

print("\n=== TRAIN baseline cross-tab ===")
print(train_comparison)
print("\n=== HOLDOUT baseline cross-tab ===")
print(holdout_comparison)

=== TRAIN profile ===
         impressions    ctr_pct  avg_position  days_since_created  n_pages
cluster                                                                   
0              213.0   0.000000      7.057069                61.0    65592
1              174.0   0.000000      7.133292               248.0    61846
2            24280.0   0.191357      5.008276               187.0     2464
3                2.0  66.666667      3.000000               225.0      335
4               60.0   0.000000     54.800000               243.0    18811

=== HOLDOUT profile ===
         impressions    ctr_pct  avg_position  days_since_created  n_pages
cluster                                                                   
0              176.0   0.000000      7.603444                71.0    10780
1              328.0   0.000000      8.204781               320.0    14160
2            25088.0   0.264874      6.434223               155.0      430
3                2.0  50.000000      3.250000        

I used K=5 (chosen over the highest-scoring K=7 for interpretability — the silhouette gain from 5→7 was small, 0.44→0.46, and fewer segments are easier for an editor to act on). Clustering was done on the 47% of training pages with zero impressions filtered out (undefined CTR/position), which I note as a real limitation — invisible pages are a separate problem needing different investigation.

The five clusters found on training data reappeared with matching shapes on the holdout set (17 unseen clients) — same impressions/position/CTR ranges, confirming these are stable, recurring patterns rather than noise from one batch.

Compared against the Week 4 baseline rule: Cluster 2 ("high-traffic underperformers," n=2,464) is mostly caught by the old rule (67% flagged review) — a good agreement check. However, Clusters 0 and 1 together (~85% of visible pages) sit on page 1 with 0% median CTR, yet the old rule only flags ~33% of them for review — clustering surfaces a large, systematic blind spot the threshold-based rule misses. Cluster 3 (n=335, 0.2% of data) shows a very high CTR (66.7%) but is built on only 1-2 clicks per page and should not be read as a genuine high-performing segment — it's flagged as statistical noise, not a real archetype.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Since clustering has no labeled "correct" groups, "errors" here mean pages sitting ambiguously between two clusters (low or negative per-point silhouette score) rather than misclassifications.

Cluster 3 (the noise cluster) has the highest average silhouette (0.671) despite being our least trustworthy segment — its 335 near-zero-impression pages are tightly bunched together mathematically, which doesn't make it a meaningful archetype, just a small, self-similar outlier group. Clusters 1, 2, and 4 show more overlap (silhouette ~0.38–0.42), meaning their boundaries are fuzzier in practice.

The 10 most ambiguous pages are dominated by cluster 4 members sitting at position 34–43 — notably better-positioned than cluster 4's typical median (~55). These pages sit genuinely on the border between "buried" (cluster 4) and "ranking on page 1" (clusters 0/1), which is intuitive: there's no hard real-world line between "barely visible" and "on page 1," so it's expected the model finds this boundary fuzzy too.

On what drives the clustering: CTR shows the largest raw spread across centroids, but this is driven entirely by the noise cluster's extreme outlier value, not a genuine signal — excluding it, position and impressions are the clearest single drivers (separating cluster 4 and cluster 2 respectively), while days_since_created contributes more evenly across multiple clusters.

In [10]:
from sklearn.metrics import silhouette_samples

# --- Part A: per-point silhouette — find the fence-sitters ---
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_train_scaled), size=5000, replace=False)
X_sample = X_train_scaled[sample_idx]
labels_sample = df_train_c["cluster"].values[sample_idx]

sil_scores = silhouette_samples(X_sample, labels_sample)

sil_df = df_train_c.iloc[sample_idx].copy()
sil_df["silhouette"] = sil_scores

print("Average silhouette per cluster (higher = more confidently separated):")
print(sil_df.groupby("cluster")["silhouette"].mean().round(3))

print("\n10 most ambiguous pages (lowest silhouette = sitting on a boundary):")
print(sil_df.nsmallest(10, "silhouette")[
    ["cluster", "impressions", "ctr_pct", "avg_position", "days_since_created", "silhouette"]
])

# --- Part B: what drives the separation? ---
centroids = pd.DataFrame(kmeans_final.cluster_centers_, columns=cluster_features)
print("\nCentroid values (in SCALED units — bigger spread = more influence on clustering):")
print(centroids.round(2))
print("\nSpread (std dev) of each feature across the 5 centroids, ranked:")
print(centroids.std().sort_values(ascending=False).round(2))

Average silhouette per cluster (higher = more confidently separated):
cluster
0    0.499
1    0.381
2    0.422
3    0.671
4    0.380
Name: silhouette, dtype: float64

10 most ambiguous pages (lowest silhouette = sitting on a boundary):
        cluster  impressions   ctr_pct  avg_position  days_since_created  \
150708        1      16060.0  0.386052     27.369054                 230   
181502        4       1328.0  0.000000     34.804970                 141   
82969         4         78.0  0.000000     40.820513                  69   
88515         4        643.0  0.155521     40.438569                  78   
190502        4         62.0  0.000000     37.758065                 110   
248982        4         26.0  0.000000     41.730769                  61   
248261        4        202.0  0.495050     40.841584                  74   
84007         4         15.0  0.000000     43.333333                  42   
190007        4         99.0  0.000000     35.818182                 134   
1583

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.